# SmartFAQSearch 🔎

**AWS AASTU Student Builders Group — Week 2**

This notebook demonstrates semantic search over a small university FAQ dataset. It uses Pandas to inspect the data, Sentence Transformers to create text embeddings, and ChromaDB to retrieve similar FAQ questions.

> Run this notebook from the `week-02` project root, where the `src/`, `data/`, and `requirements.txt` items are located. The existing application remains the source of truth for the search implementation.


## Learning objectives

By the end of this notebook, you should be able to:

- Load and inspect FAQ data from a CSV file.
- Explain what text embeddings represent.
- Build a ChromaDB search index using the project's existing functions.
- Try relevant and out-of-scope questions.
- Interpret a small evaluation result and recognize limitations.


## 1. Install dependencies (if needed)

If you use the project's existing virtual environment, install the requirements from PowerShell in the project root:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

If you run this notebook in a separate environment, uncomment and run the next cell. The first model run may download the embedding model and requires internet access.


In [ ]:
# Uncomment only if your notebook environment needs the dependencies installed.
# %pip install pandas chromadb sentence-transformers

## 2. Import libraries and project functions

The notebook imports the same functions used by the command-line application so that the notebook and application stay consistent.


In [ ]:
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer

from src.SmartFAQSearch import (
    load_faqs,
    build_search_index,
    search_faqs,
    MODEL_NAME,
)

PROJECT_ROOT = Path.cwd()
print(f"Notebook working directory: {PROJECT_ROOT}")
print(f"Embedding model: {MODEL_NAME}")

### Check the working directory

The working directory should be the `week-02` folder. If the next cell cannot find `data/faqs.csv` or `src/SmartFAQSearch.py`, change the notebook's working directory to the project root before continuing.


In [ ]:
data_path = PROJECT_ROOT / "data" / "faqs.csv"
source_path = PROJECT_ROOT / "src" / "SmartFAQSearch.py"

print("FAQ CSV exists:", data_path.exists())
print("Application source exists:", source_path.exists())
assert data_path.exists(), "Run this notebook from the week-02 project root."
assert source_path.exists(), "The src/SmartFAQSearch.py file was not found."

## 3. Load and inspect the FAQ dataset

First, inspect the raw CSV with Pandas. This helps us understand the data before indexing it.


In [ ]:
raw_faqs = pd.read_csv(data_path)
print(f"Rows: {len(raw_faqs)}")
print("Columns:", raw_faqs.columns.tolist())
display(raw_faqs.head(10))

### Use the project's data-loading function

`load_faqs()` applies the validation and cleaning logic implemented in the application. We inspect its output before creating the search index.


In [ ]:
faqs = load_faqs()
print(f"FAQs available for indexing: {len(faqs)}")
display(faqs.head())

## 4. Load the embedding model

Sentence Transformers converts text into numerical vectors called **embeddings**. Questions with related meanings may have vectors that are close to one another, which helps semantic search find matches even when wording differs.

The model below is the same model configured in the application. Its first load may take time because the model needs to be downloaded.


In [ ]:
model = SentenceTransformer(MODEL_NAME)
print("Embedding model loaded successfully.")

## 5. Build the ChromaDB search index

ChromaDB stores the FAQ embeddings and supports similarity search. We use the project's existing `build_search_index()` function rather than duplicating the indexing implementation in the notebook.


In [ ]:
collection = build_search_index(faqs, model)
print("Search index created.")

## 6. Try semantic search

The `search_faqs()` function applies the application's configured relevance threshold and ambiguity handling. The output format follows the implementation in `src/SmartFAQSearch.py`.


In [ ]:
def try_question(question: str, top_k: int = 3):
    """Run one question through the existing semantic-search function."""
    print(f"Question: {question}")
    result = search_faqs(question, collection, model, top_k=top_k)
    print("Result:")
    print(result)
    print("-" * 70)
    return result

### Example A — a relevant question

This question asks where a student can find grades. Compare the returned FAQ with the question to see whether the result answers the same information need.


In [ ]:
relevant_result = try_question("Where can I find my grades?")

### Example B — different wording

Semantic search is useful when the wording is different from the wording stored in the FAQ dataset. Review the returned match rather than assuming that a high similarity automatically means the answer is correct.


In [ ]:
paraphrase_result = try_question("How do I see my exam results?")

### Example C — an out-of-scope question

This question is unrelated to university FAQs. The application should reject it when its closest match falls outside the configured relevance threshold.


In [ ]:
out_of_scope_result = try_question("How do I repair a car?")

## 7. Evaluation results and interpretation

The project's recorded evaluation used a distance threshold of `0.65` and a small, manually selected set of questions. Its reported results were:

| Metric                                       |        Result |
| -------------------------------------------- | ------------: |
| Relevant queries matched to the expected FAQ | 13/14 (92.9%) |
| Relevant queries incorrectly rejected        |          0/14 |
| Out-of-scope queries rejected                |    8/8 (100%) |
| False acceptances in the out-of-scope set    |           0/8 |

These figures are the previously recorded evaluation results, not results generated automatically by this notebook. Run the evaluation script to reproduce the current evaluation.


In [ ]:
# Run the project's separate real-model evaluation script from PowerShell:
# .\.venv\Scripts\python.exe .\tests\evaluate_search.py

### Known limitation

In the recorded evaluation, **“How do I see my exam results?”** retrieved **“When are examination results published?”** instead of the expected FAQ **“Where can I find my grades?”**. The questions are related, but they represent different information needs.

This illustrates an important point: embedding similarity is not a guarantee of answer correctness. A larger and more diverse evaluation set is needed before drawing strong conclusions about search quality.


## 8. Reflection Questions

The answers below use the behavior and examples observed in this SmartFAQSearch project.

### 1. How is semantic search different from exact keyword matching?

<p style="font-size: 16px;">Exact keyword matching looks for the same words that appear in stored questions. Semantic search looks for similar meaning. For example, a student might ask “Where can I check my marks?” while the FAQ says “Where can I find my grades?” Semantic search can match these questions even though the wording is different.</p>

### 2. What is an embedding, and why is it useful here?

<p style="font-size: 16px;">An embedding is a numerical representation of text. In this project, Sentence Transformers converts FAQ questions into numerical vectors. The system compares these vectors to find questions with related meanings, which helps it handle different ways of asking the same thing.</p>

### 3. What role does ChromaDB play in this project?

<p style="font-size: 16px;">ChromaDB stores the FAQ embeddings and helps retrieve questions similar to a user's query. It allows the application to search the indexed FAQs and return the closest matches instead of relying only on exact word matches.</p>

### 4. Why should the application reject some questions?

<p style="font-size: 16px;">The application should reject questions that are not relevant to the university FAQs. For example, if someone asks how to repair a car engine, the system should not return an unrelated university answer just because it is the closest available match. This project uses a distance threshold of `0.65` to help decide whether a result is relevant enough to accept.</p>

### 5. Why can a semantically similar result still be the wrong answer?

<p style="font-size: 16px;">Two questions can be related but still ask for different information. In our evaluation, “How do I see my exam results?” retrieved “When are examination results published?” instead of the expected “Where can I find my grades?” One asks when results are published, while the other asks where to find grades. This shows that similarity alone does not guarantee a correct answer.</p>

### 6. What additional test questions would help you evaluate the system more reliably?

<p style="font-size: 16px;">I would test more ways of asking the same question, including short wording, spelling mistakes, and paraphrases. Examples include “Where are my marks posted?”, “How can I view my grades online?”, and “When will my exam results come out?” I would also test unrelated questions about programming, weather, travel, and other topics. A larger and more varied test set would help reveal when the system accepts a wrong match or rejects a relevant question.</p>


## Conclusion

This notebook demonstrates the main workflow behind SmartFAQSearch: inspect FAQ data, create text embeddings, index them in ChromaDB, retrieve semantically similar questions, and evaluate both relevant and out-of-scope queries. The application is a learning project and its sample FAQ data does not represent verified university policies.
